# 01 - Analisis Exploratorio de Datos

## 1. Contexto del problema
Este notebook corresponde al caso **Logistica 4.0: Prediccion de Tiempos de Entrega (Last Mile)**.

Objetivo de esta etapa:
- Explorar la calidad del dataset.
- Detectar problemas de limpieza y consistencia.
- Analizar el comportamiento de la variable objetivo `target_tiempo_entrega`.
- Preparar datos y funciones reutilizables para modelado posterior.

Este notebook **solo realiza EDA y limpieza exploratoria**. No entrena modelos finales ni ejecuta optimizacion de hiperparametros.


## 2. Importacion de librerias y modulos


In [8]:
from pathlib import Path
import shutil
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

for relative_dir in [
    "data/raw",
    "data/processed",
    "results/metrics",
    "results/plots",
    "results/reports",
    "docs",
]:
    (PROJECT_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)

from src import data_loading, data_preprocessing, exploratory_analysis, feature_engineering, visualization

importlib.reload(data_loading)
importlib.reload(data_preprocessing)
importlib.reload(exploratory_analysis)
importlib.reload(feature_engineering)
importlib.reload(visualization)

print(f"Project root: {PROJECT_ROOT}")


Project root: E:\projectsWindows\scy1101-ev2-logistica-ml


## 3. Carga del dataset


In [9]:
def resolve_dataset_path() -> Path:
    candidates = [
        PROJECT_ROOT / "data" / "raw" / "5_logistica_40.csv",
        Path.home() / "Descargas" / "5_logistica_40.csv",
        Path.home() / "Downloads" / "5_logistica_40.csv",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "No se encontro 5_logistica_40.csv en data/raw ni en Descargas/Downloads. "
        f"Rutas revisadas: {candidates}"
    )


dataset_path = resolve_dataset_path()
project_raw_path = PROJECT_ROOT / "data" / "raw" / "5_logistica_40.csv"

if dataset_path != project_raw_path and not project_raw_path.exists():
    shutil.copy2(dataset_path, project_raw_path)
    dataset_path = project_raw_path

print(f"Ruta dataset usada: {dataset_path}")
df = data_loading.load_dataset(str(dataset_path))

print("Shape:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())
display(df.tail())
df.info()


Ruta dataset usada: E:\projectsWindows\scy1101-ev2-logistica-ml\data\raw\5_logistica_40.csv
Shape: (1200, 12)
Columnas: ['distancia_km', 'trafico_nivel', 'clima', 'tipo_vehiculo', 'peso_carga_kg', 'paradas_previas', 'experiencia_chofer_anios', 'hora_despacho', 'costo_envio', 'consumo_nafta', 'id_bodega', 'target_tiempo_entrega']


,distancia_km,trafico_nivel,clima,tipo_vehiculo,peso_carga_kg,paradas_previas,experiencia_chofer_anios,hora_despacho,costo_envio,consumo_nafta,id_bodega,target_tiempo_entrega
0,2.810680,Alto,Lluvia,Camion,177.287994,7,7,11,5836.503551,6.997771,4,63.577071
1,25.182597,Medio,Soleado,Moto,43.785237,9,13,8,13321.428789,5.378228,2,50.996473
2,9.661996,Medio,Nublado,Moto,291.553925,6,6,16,7251.865187,9.165121,4,41.474435
3,9.665324,Bajo,Lluvia,Moto,353.772316,6,2,20,8693.301465,12.892030,3,64.123619
4,27.116790,Medio,Lluvia,Camion,291.044978,7,22,15,4383.469231,14.519544,4,36.767529


,distancia_km,trafico_nivel,clima,tipo_vehiculo,peso_carga_kg,paradas_previas,experiencia_chofer_anios,hora_despacho,costo_envio,consumo_nafta,id_bodega,target_tiempo_entrega
1195,21.356220,Bajo,Soleado,Van,427.141198,4,17,9,13360.119667,10.704305,4,34.884870
1196,34.942634,Bajo,Nublado,Camion,16.751939,5,1,12,2429.969915,5.713174,3,57.752377
1197,48.191085,Critico,Nublado,Camion,434.345189,6,13,14,8952.053702,11.623526,4,64.032767
1198,15.132231,Bajo,Lluvia,Moto,443.734312,9,6,12,4733.343779,6.472894,1,33.613025
1199,47.810999,Alto,Nublado,Moto,223.703539,8,5,16,6785.986289,7.796994,2,31.611671


<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   distancia_km              1200 non-null   float64
 1   trafico_nivel             1200 non-null   str    
 2   clima                     1200 non-null   str    
 3   tipo_vehiculo             1200 non-null   str    
 4   peso_carga_kg             1200 non-null   float64
 5   paradas_previas           1200 non-null   int64  
 6   experiencia_chofer_anios  1200 non-null   int64  
 7   hora_despacho             1200 non-null   int64  
 8   costo_envio               1200 non-null   float64
 9   consumo_nafta             1200 non-null   float64
 10  id_bodega                 1200 non-null   int64  
 11  target_tiempo_entrega     1200 non-null   float64
dtypes: float64(5), int64(4), str(3)
memory usage: 112.6 KB


## 4. Vista general del dataset


In [10]:
overview_table = exploratory_analysis.dataset_overview(df)
column_types_table = exploratory_analysis.column_types_summary(df)

display(overview_table)
display(column_types_table)


,filas,columnas,memoria_mb,columnas_numericas,cantidad_numericas,columnas_categoricas,cantidad_categoricas,target_detectado
0,1200,12,0.2691,"distancia_km, peso_carga_kg, paradas_previas, ...",9,"trafico_nivel, clima, tipo_vehiculo",3,target_tiempo_entrega


,columna,dtype,tipo_semantico
0,distancia_km,float64,numerica
1,trafico_nivel,str,categorica
2,clima,str,categorica
3,tipo_vehiculo,str,categorica
4,peso_carga_kg,float64,numerica
5,paradas_previas,int64,numerica
6,experiencia_chofer_anios,int64,numerica
7,hora_despacho,int64,numerica
8,costo_envio,float64,numerica
9,consumo_nafta,float64,numerica


## 5. Descripcion de columnas y rol dentro del proyecto


In [11]:
column_roles = pd.DataFrame(
    [
        ["distancia_km", "Numerica", "Distancia recorrida en km", "Feature numerica"],
        ["trafico_nivel", "Categorica/Ordinal", "Nivel de trafico en la ruta", "Feature categorica/ordinal"],
        ["clima", "Categorica", "Condicion climatica del despacho", "Feature categorica"],
        ["tipo_vehiculo", "Categorica", "Tipo de vehiculo de reparto", "Feature categorica"],
        ["peso_carga_kg", "Numerica", "Peso de la carga en kilogramos", "Feature numerica"],
        ["paradas_previas", "Numerica discreta", "Cantidad de paradas antes de la entrega", "Feature numerica discreta"],
        ["experiencia_chofer_anios", "Numerica discreta", "Experiencia del chofer en anios", "Feature numerica discreta"],
        ["hora_despacho", "Temporal", "Hora de salida del despacho", "Feature temporal"],
        ["costo_envio", "Numerica", "Costo del envio", "Feature numerica"],
        ["consumo_nafta", "Numerica", "Consumo de combustible", "Feature con posible leakage"],
        ["id_bodega", "Categorica (ID)", "Identificador de bodega", "Feature categorica"],
        ["target_tiempo_entrega", "Numerica continua", "Tiempo final de entrega (minutos)", "Target continuo"],
    ],
    columns=["columna", "tipo", "descripcion", "rol_sugerido"],
)

display(column_roles)


,columna,tipo,descripcion,rol_sugerido
0,distancia_km,Numerica,Distancia recorrida en km,Feature numerica
1,trafico_nivel,Categorica/Ordinal,Nivel de trafico en la ruta,Feature categorica/ordinal
2,clima,Categorica,Condicion climatica del despacho,Feature categorica
3,tipo_vehiculo,Categorica,Tipo de vehiculo de reparto,Feature categorica
4,peso_carga_kg,Numerica,Peso de la carga en kilogramos,Feature numerica
5,paradas_previas,Numerica discreta,Cantidad de paradas antes de la entrega,Feature numerica discreta
6,experiencia_chofer_anios,Numerica discreta,Experiencia del chofer en anios,Feature numerica discreta
7,hora_despacho,Temporal,Hora de salida del despacho,Feature temporal
8,costo_envio,Numerica,Costo del envio,Feature numerica
9,consumo_nafta,Numerica,Consumo de combustible,Feature con posible leakage


## 6. Analisis de valores nulos


In [12]:
missing_table = exploratory_analysis.missing_values_table(df)
display(missing_table)

if missing_table["cantidad_nulos"].sum() == 0:
    display(Markdown("**Conclusion:** No se detectan valores nulos en el dataset original. No se requiere imputacion inicial, pero se mantiene validacion en el pipeline por reproducibilidad."))


,columna,cantidad_nulos,porcentaje_nulos
0,clima,0,0.0
1,consumo_nafta,0,0.0
2,costo_envio,0,0.0
3,distancia_km,0,0.0
4,experiencia_chofer_anios,0,0.0
5,hora_despacho,0,0.0
6,id_bodega,0,0.0
7,paradas_previas,0,0.0
8,peso_carga_kg,0,0.0
9,target_tiempo_entrega,0,0.0


**Conclusion:** No se detectan valores nulos en el dataset original. No se requiere imputacion inicial, pero se mantiene validacion en el pipeline por reproducibilidad.

## 7. Analisis de duplicados


In [13]:
duplicates_table = exploratory_analysis.duplicated_summary(df)
display(duplicates_table)

if int(duplicates_table.loc[0, "duplicados_exactos"]) == 0:
    display(Markdown("**Conclusion:** No se detectan duplicados exactos."))


,filas_totales,duplicados_exactos,porcentaje_duplicados
0,1200,0,0.0


**Conclusion:** No se detectan duplicados exactos.

## 8. Analisis de variables categoricas


In [14]:
df_base = data_preprocessing.normalize_column_names(df)
df_base = data_preprocessing.clean_text_columns(df_base, data_preprocessing.TEXT_CATEGORICAL_COLUMNS)
df_base = data_preprocessing.convert_id_bodega_to_category(df_base)

categorical_columns = ["trafico_nivel", "clima", "tipo_vehiculo", "id_bodega"]
categorical_counts_table = exploratory_analysis.categorical_value_counts(df_base, categorical_columns)
display(categorical_counts_table)

for col in categorical_columns:
    if col in df_base.columns:
        print(f"Categorias en {col}:", sorted(df_base[col].astype(str).unique().tolist()))

plots_dir = PROJECT_ROOT / "results" / "plots"
categorical_plot_paths = visualization.plot_categorical_counts(df_base, categorical_columns, plots_dir)
print("Graficos categoricos guardados:")
for path in categorical_plot_paths:
    print("-", path)


,columna,categoria,cantidad,porcentaje
0,trafico_nivel,Critico,317,26.42
1,trafico_nivel,Medio,302,25.17
2,trafico_nivel,Bajo,293,24.42
3,trafico_nivel,Alto,288,24.00
4,clima,Lluvia,425,35.42
5,clima,Soleado,390,32.50
6,clima,Nublado,385,32.08
7,tipo_vehiculo,Moto,413,34.42
8,tipo_vehiculo,Van,401,33.42
9,tipo_vehiculo,Camion,386,32.17


Categorias en trafico_nivel: ['Alto', 'Bajo', 'Critico', 'Medio']
Categorias en clima: ['Lluvia', 'Nublado', 'Soleado']
Categorias en tipo_vehiculo: ['Camion', 'Moto', 'Van']
Categorias en id_bodega: ['1', '2', '3', '4']


E:\projectsWindows\scy1101-ev2-logistica-ml\src\visualization.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x=column, order=order, ax=ax, palette="Blues_d")
E:\projectsWindows\scy1101-ev2-logistica-ml\src\visualization.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x=column, order=order, ax=ax, palette="Blues_d")
E:\projectsWindows\scy1101-ev2-logistica-ml\src\visualization.py:119: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, x=column, order=order, ax=ax, palette="Blues_d")
E:\projectsWindows\

Graficos categoricos guardados:
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\count_trafico_nivel.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\count_clima.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\count_tipo_vehiculo.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\count_id_bodega.png


Conclusiones esperadas de categoricas:
- `trafico_nivel` tiene orden logico: Bajo < Medio < Alto < Critico.
- `clima` y `tipo_vehiculo` no tienen orden natural.
- `id_bodega` debe tratarse como variable categorica, no continua.


## 9. Analisis de variables numericas


In [15]:
numeric_columns = [
    "distancia_km",
    "peso_carga_kg",
    "paradas_previas",
    "experiencia_chofer_anios",
    "hora_despacho",
    "costo_envio",
    "consumo_nafta",
    "target_tiempo_entrega",
]

numeric_table = exploratory_analysis.numeric_summary(df_base, numeric_columns)
display(numeric_table)

hist_paths = visualization.plot_numeric_histograms(df_base, numeric_columns, plots_dir)
box_paths = visualization.plot_boxplots(df_base, numeric_columns, plots_dir)
print("Histogramas guardados:", len(hist_paths))
print("Boxplots guardados:", len(box_paths))


,columna,count,mean,std,min,25%,50%,75%,max
0,distancia_km,1200.0,25.625439,13.969398,1.006319,13.824427,25.190264,37.759237,49.916692
1,peso_carga_kg,1200.0,252.360787,146.530933,1.004208,125.824913,253.954866,384.908156,499.859144
2,paradas_previas,1200.0,4.522500,2.872229,0.000000,2.000000,4.000000,7.000000,9.000000
3,experiencia_chofer_anios,1200.0,12.354167,7.030642,1.000000,6.000000,12.000000,18.000000,24.000000
4,hora_despacho,1200.0,13.563333,4.001791,7.000000,10.000000,14.000000,17.000000,20.000000
5,costo_envio,1200.0,8405.057643,3745.265400,2008.186755,5319.279172,8171.170721,11700.600263,14992.916035
6,consumo_nafta,1200.0,10.011712,2.811400,5.013236,7.643946,9.875638,12.314524,14.996899
7,target_tiempo_entrega,1200.0,45.624603,14.854897,-3.996755,35.437966,45.605611,55.587505,89.965449


Histogramas guardados: 8
Boxplots guardados: 8


## 10. Validacion de reglas de negocio


In [16]:
business_rules_table = exploratory_analysis.business_rules_report(df_base)
display(business_rules_table)

invalid_target_rows = df_base.loc[pd.to_numeric(df_base["target_tiempo_entrega"], errors="coerce") <= 0]

print("Registros con target <= 0:", len(invalid_target_rows))
display(
    invalid_target_rows[
        ["distancia_km", "trafico_nivel", "clima", "tipo_vehiculo", "target_tiempo_entrega"]
    ]
)


,regla,cantidad_incumplimientos,porcentaje_incumplimientos,accion_recomendada
0,distancia_km > 0,0,0.00,Sin accion correctiva
1,peso_carga_kg > 0,0,0.00,Sin accion correctiva
2,paradas_previas >= 0,0,0.00,Sin accion correctiva
3,experiencia_chofer_anios >= 0,0,0.00,Sin accion correctiva
4,hora_despacho entre 0 y 23,0,0.00,Sin accion correctiva
5,costo_envio > 0,0,0.00,Sin accion correctiva
6,consumo_nafta > 0,0,0.00,Sin accion correctiva
7,target_tiempo_entrega > 0,2,0.17,Eliminar registros invalidos para dataset limpio
8,id_bodega dentro del rango observado,0,0.00,Sin accion correctiva


Registros con target <= 0: 2


,distancia_km,trafico_nivel,clima,tipo_vehiculo,target_tiempo_entrega
806,32.243258,Critico,Lluvia,Moto,-3.124278
1153,43.130505,Alto,Soleado,Camion,-3.996755


**Conclusion obligatoria:**
Los registros con `target_tiempo_entrega` menor o igual a cero se consideran invalidos, porque una entrega no puede tener duracion negativa o nula. Se recomienda eliminarlos del dataset limpio.


## 11. Analisis especifico del target


In [17]:
target_table = exploratory_analysis.target_summary(df_base, "target_tiempo_entrega")
display(target_table)

df_flagged = data_preprocessing.add_suspicious_time_flag(df_base, threshold=10)

target_invalid_table = df_flagged.loc[pd.to_numeric(df_flagged["target_tiempo_entrega"], errors="coerce") <= 0]
target_suspicious_table = df_flagged.loc[pd.to_numeric(df_flagged["target_tiempo_entrega"], errors="coerce") < 10]

print("Registros con target <= 0:", len(target_invalid_table))
print("Registros con target < 10:", len(target_suspicious_table))

display(target_invalid_table[["distancia_km", "trafico_nivel", "target_tiempo_entrega"]])
display(target_suspicious_table[["distancia_km", "trafico_nivel", "target_tiempo_entrega", "tiempo_sospechoso"]].head(20))

target_distribution_path = visualization.plot_target_distribution(
    df_flagged,
    target_column="target_tiempo_entrega",
    output_path=plots_dir / "target_distribution.png",
)
suspicious_plot_path = visualization.plot_suspicious_times(
    df_flagged,
    output_path=plots_dir / "target_suspicious_times.png",
)
print("Grafico target:", target_distribution_path)
print("Grafico sospechosos:", suspicious_plot_path)


,metrica,valor
0,min,-3.996755
1,max,89.965449
2,media,45.624603
3,mediana,45.605611
4,desviacion_estandar,14.854897
5,q1,35.437966
6,q3,55.587505
7,iqr,20.149539
8,limite_inferior_iqr,5.213658
9,limite_superior_iqr,85.811813


Registros con target <= 0: 2
Registros con target < 10: 10


,distancia_km,trafico_nivel,target_tiempo_entrega
806,32.243258,Critico,-3.124278
1153,43.130505,Alto,-3.996755


,distancia_km,trafico_nivel,target_tiempo_entrega,tiempo_sospechoso
282,22.602119,Critico,5.140510,True
806,32.243258,Critico,-3.124278,True
862,32.851722,Bajo,8.518243,True
876,29.394702,Alto,8.587269,True
888,10.206618,Medio,8.561709,True
992,19.731318,Bajo,5.037872,True
995,1.752881,Alto,9.775115,True
1074,24.306103,Medio,6.937210,True
1134,10.816733,Bajo,7.460871,True
1153,43.130505,Alto,-3.996755,True


Grafico target: E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\target_distribution.png
Grafico sospechosos: E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\target_suspicious_times.png


Criterio aplicado:
- `target <= 0`: invalido, se elimina del dataset limpio.
- `target < 10`: se marca como sospechoso, pero no se elimina automaticamente.


## 12. Deteccion de outliers con IQR


In [18]:
outliers_iqr_table = exploratory_analysis.detect_outliers_iqr(df_base, numeric_columns)
display(outliers_iqr_table)

target_outlier_info = outliers_iqr_table.loc[outliers_iqr_table["columna"] == "target_tiempo_entrega"]
display(target_outlier_info)


,columna,q1,q3,iqr,limite_inferior,limite_superior,cantidad_outliers,porcentaje_outliers
0,distancia_km,13.824427,37.759237,23.934810,-22.077788,73.661452,0,0.0
1,peso_carga_kg,125.824913,384.908156,259.083243,-262.799952,773.533021,0,0.0
2,paradas_previas,2.000000,7.000000,5.000000,-5.500000,14.500000,0,0.0
3,experiencia_chofer_anios,6.000000,18.000000,12.000000,-12.000000,36.000000,0,0.0
4,hora_despacho,10.000000,17.000000,7.000000,-0.500000,27.500000,0,0.0
5,costo_envio,5319.279172,11700.600263,6381.321090,-4252.702463,21272.581898,0,0.0
6,consumo_nafta,7.643946,12.314524,4.670578,0.638078,19.320391,0,0.0
7,target_tiempo_entrega,35.437966,55.587505,20.149539,5.213658,85.811813,6,0.5


,columna,q1,q3,iqr,limite_inferior,limite_superior,cantidad_outliers,porcentaje_outliers
7,target_tiempo_entrega,35.437966,55.587505,20.149539,5.213658,85.811813,6,0.5


Interpretacion:
En logistica, un tiempo alto puede ser un evento real y relevante; por eso los outliers se revisan y documentan antes de decidir si eliminarlos.


## 13. Correlacion con el target


In [19]:
corr_matrix = df_base[numeric_columns].corr(numeric_only=True)
display(corr_matrix)

corr_target_table = exploratory_analysis.correlation_with_target(df_base[numeric_columns], "target_tiempo_entrega")
display(corr_target_table)

corr_heatmap_path = visualization.plot_correlation_heatmap(
    df_base[numeric_columns],
    output_path=plots_dir / "correlation_heatmap.png",
)
print("Heatmap guardado en:", corr_heatmap_path)


,distancia_km,peso_carga_kg,paradas_previas,experiencia_chofer_anios,hora_despacho,costo_envio,consumo_nafta,target_tiempo_entrega
distancia_km,1.000000,-0.027254,0.010507,-0.001722,0.039988,0.032168,-0.025122,-0.000397
peso_carga_kg,-0.027254,1.000000,-0.032290,0.024442,0.011619,-0.000099,0.042905,-0.010982
paradas_previas,0.010507,-0.032290,1.000000,-0.023958,-0.015616,-0.026416,-0.008405,-0.046781
experiencia_chofer_anios,-0.001722,0.024442,-0.023958,1.000000,0.018811,0.057648,-0.004767,-0.008998
hora_despacho,0.039988,0.011619,-0.015616,0.018811,1.000000,0.023421,0.058587,-0.035751
costo_envio,0.032168,-0.000099,-0.026416,0.057648,0.023421,1.000000,0.017528,0.002124
consumo_nafta,-0.025122,0.042905,-0.008405,-0.004767,0.058587,0.017528,1.000000,-0.020673
target_tiempo_entrega,-0.000397,-0.010982,-0.046781,-0.008998,-0.035751,0.002124,-0.020673,1.000000


,columna,correlacion,correlacion_absoluta
0,paradas_previas,-0.046781,0.046781
1,hora_despacho,-0.035751,0.035751
2,consumo_nafta,-0.020673,0.020673
3,peso_carga_kg,-0.010982,0.010982
4,experiencia_chofer_anios,-0.008998,0.008998
5,costo_envio,0.002124,0.002124
6,distancia_km,-0.000397,0.000397


Heatmap guardado en: E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\correlation_heatmap.png


Interpretacion:
Una baja correlacion lineal no significa que no exista relacion, pero si sugiere que el target podria depender de interacciones no lineales, variables categoricas o variables que no estan presentes en el dataset.

Variables futuras sugeridas:
- comuna o zona de entrega
- coordenadas o distancia real por ruta
- dia de la semana
- fecha
- ventana horaria real
- densidad urbana
- prioridad del pedido
- estado del repartidor
- historial de congestion
- tipo de ruta
- cantidad de pedidos simultaneos


## 14. Analisis target vs categoricas


In [20]:
target_by_category_paths = visualization.plot_target_by_category(
    df_flagged,
    target_column="target_tiempo_entrega",
    categorical_columns=categorical_columns,
    output_dir=plots_dir,
)
print("Graficos target vs categoricas guardados:")
for path in target_by_category_paths:
    print("-", path)


Graficos target vs categoricas guardados:
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\target_by_trafico_nivel.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\target_by_clima.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\target_by_tipo_vehiculo.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\target_by_id_bodega.png


## 15. Analisis target vs numericas


In [21]:
numeric_feature_columns = [col for col in numeric_columns if col != "target_tiempo_entrega"]

scatter_paths = visualization.plot_scatter_numeric_vs_target(
    df_flagged,
    numeric_columns=numeric_feature_columns,
    target_column="target_tiempo_entrega",
    output_dir=plots_dir,
)

print("Graficos scatter guardados:")
for path in scatter_paths:
    print("-", path)

# Comparacion exploratoria antes y despues de StandardScaler (sin modelado)
scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_flagged[numeric_feature_columns])
scaled_df = pd.DataFrame(scaled_array, columns=numeric_feature_columns)

scaling_comparison = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "media_original": df_flagged[numeric_feature_columns].mean().values,
        "std_original": df_flagged[numeric_feature_columns].std(ddof=0).values,
        "media_escalada": scaled_df.mean().values,
        "std_escalada": scaled_df.std(ddof=0).values,
    }
)

display(scaling_comparison)


Graficos scatter guardados:
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_distancia_km_vs_target_tiempo_entrega.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_peso_carga_kg_vs_target_tiempo_entrega.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_paradas_previas_vs_target_tiempo_entrega.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_experiencia_chofer_anios_vs_target_tiempo_entrega.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_hora_despacho_vs_target_tiempo_entrega.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_costo_envio_vs_target_tiempo_entrega.png
- E:\projectsWindows\scy1101-ev2-logistica-ml\results\plots\scatter_consumo_nafta_vs_target_tiempo_entrega.png


,feature,media_original,std_original,media_escalada,std_escalada
0,distancia_km,25.625439,13.963577,-1.369275e-17,1.0
1,peso_carga_kg,252.360787,146.469866,-1.125026e-16,1.0
2,paradas_previas,4.522500,2.871032,1.480297e-17,1.0
3,experiencia_chofer_anios,12.354167,7.027712,7.993606e-17,1.0
4,hora_despacho,13.563333,4.000124,1.998401e-16,1.0
5,costo_envio,8405.057643,3743.704547,2.960595e-17,1.0
6,consumo_nafta,10.011712,2.810228,4.381680e-16,1.0


Nota de escalado:
- KNN, K-Means, PCA o SVR suelen requerir escalado (ej. `StandardScaler`).
- Arboles de decision y Random Forest no lo requieren estrictamente, aunque puede mantenerse por orden en pipelines.


## 16. Feature engineering exploratorio


In [22]:
df_exploratory_features = feature_engineering.prepare_exploratory_features(df_flagged)

new_feature_columns = [
    "hora_sin",
    "hora_cos",
    "costo_por_km",
    "consumo_por_km",
    "carga_por_km",
    "paradas_por_km",
]

display(df_exploratory_features[new_feature_columns].head())
display(df_exploratory_features[new_feature_columns].describe().T)

feature_corr_table = exploratory_analysis.correlation_with_target(
    df_exploratory_features[numeric_columns + new_feature_columns],
    target_column="target_tiempo_entrega",
)
display(feature_corr_table)

display(feature_engineering.get_feature_recommendations())


,hora_sin,hora_cos,costo_por_km,consumo_por_km,carga_por_km,paradas_por_km
0,0.258819,-0.965926,2076.545294,2.489708,63.076557,2.490501
1,0.866025,-0.500000,528.993450,0.213569,1.738710,0.357390
2,-0.866025,-0.500000,750.555564,0.948574,30.175329,0.620990
3,-0.866025,0.500000,899.431941,1.333844,36.602219,0.620776
4,-0.707107,-0.707107,161.651479,0.535445,10.733017,0.258143


,count,mean,std,min,25%,50%,75%,max
hora_sin,1200.0,-0.216489,0.707224,-1.000000,-0.866025,-0.500000,0.500000,0.965926
hora_cos,1200.0,-0.490395,0.461598,-1.000000,-0.866025,-0.707107,-0.258819,0.500000
costo_por_km,1200.0,638.473743,998.408553,41.829664,198.240520,329.214012,606.152284,9080.352090
consumo_por_km,1200.0,0.769868,1.160469,0.109381,0.255615,0.387042,0.740691,11.073719
carga_por_km,1200.0,20.160742,36.419180,0.022575,4.645219,9.768108,18.568006,334.935545
paradas_por_km,1200.0,0.334763,0.593247,0.000000,0.080109,0.177087,0.338270,8.943489


,columna,correlacion,correlacion_absoluta
0,paradas_previas,-0.046781,0.046781
1,hora_cos,-0.044914,0.044914
2,hora_despacho,-0.035751,0.035751
3,hora_sin,0.023763,0.023763
4,paradas_por_km,-0.023452,0.023452
5,consumo_nafta,-0.020673,0.020673
6,peso_carga_kg,-0.010982,0.010982
7,experiencia_chofer_anios,-0.008998,0.008998
8,carga_por_km,0.005380,0.005380
9,costo_envio,0.002124,0.002124


,feature,tipo_transformacion,recomendacion
0,hora_sin,Ciclica,Usar junto a hora_cos para respetar periodicid...
1,hora_cos,Ciclica,Complementa hora_sin para capturar comportamie...
2,costo_por_km,Ratio,Aproxima intensidad de costo por unidad de dis...
3,consumo_por_km,Ratio,Puede capturar eficiencia de ruta y vehiculo.
4,carga_por_km,Ratio,Relaciona exigencia de carga con distancia rec...
5,paradas_por_km,Ratio,Representa densidad de paradas por trayecto.
6,consumo_nafta,Riesgo de leakage,Comparar modelos con y sin esta variable por p...


## 17. Revision de posible leakage


La variable `consumo_nafta` puede tener dos interpretaciones:

- Si representa consumo estimado antes del despacho, puede utilizarse como predictor.
- Si representa consumo real observado despues de completar la entrega, seria fuga de informacion, porque no estaria disponible al momento de predecir.

Por lo tanto, en modelado posterior se recomienda comparar dos experimentos:
1. Modelo con `consumo_nafta`.
2. Modelo sin `consumo_nafta`.


## 18. Dataset limpio preliminar


In [23]:
df_clean, cleaning_summary = data_preprocessing.clean_logistics_dataset(df, remove_invalid_target=True)
df_clean_features = feature_engineering.prepare_exploratory_features(df_clean)

print("Resumen de limpieza:")
for key, value in cleaning_summary.items():
    if key in {"business_rules_report", "invalid_target_rows"}:
        continue
    print(f"- {key}: {value}")

if not cleaning_summary["invalid_target_rows"].empty:
    print("
Registros removidos por target invalido:")
    display(cleaning_summary["invalid_target_rows"][["distancia_km", "trafico_nivel", "target_tiempo_entrega"]])

processed_clean_path = PROJECT_ROOT / "data" / "processed" / "logistica_limpio_eda.csv"
processed_features_path = PROJECT_ROOT / "data" / "processed" / "logistica_limpio_eda_features.csv"

df_clean.to_csv(processed_clean_path, index=False)
df_clean_features.to_csv(processed_features_path, index=False)

print("Dataset limpio guardado en:", processed_clean_path)
print("Dataset limpio + features guardado en:", processed_features_path)


SyntaxError: unterminated string literal (detected at line 11) (2824605190.py, line 11)

## 19. Exportacion de reportes


In [ ]:
reports_dir = PROJECT_ROOT / "results" / "reports"
metrics_dir = PROJECT_ROOT / "results" / "metrics"
reports_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

overview_table.to_csv(reports_dir / "eda_dataset_overview.csv", index=False)
missing_table.to_csv(reports_dir / "eda_missing_values.csv", index=False)
duplicates_table.to_csv(reports_dir / "eda_duplicates.csv", index=False)
categorical_counts_table.to_csv(reports_dir / "eda_categorical_counts.csv", index=False)
numeric_table.to_csv(reports_dir / "eda_numeric_summary.csv", index=False)
business_rules_table.to_csv(reports_dir / "eda_business_rules_report.csv", index=False)
outliers_iqr_table.to_csv(reports_dir / "eda_outliers_iqr.csv", index=False)
corr_target_table.to_csv(reports_dir / "eda_correlations_target.csv", index=False)
target_suspicious_table.to_csv(reports_dir / "eda_suspicious_times.csv", index=False)
scaling_comparison.to_csv(metrics_dir / "eda_scaling_comparison.csv", index=False)

for category_col in categorical_columns:
    category_df = categorical_counts_table[categorical_counts_table["columna"] == category_col]
    category_df.to_csv(reports_dir / f"eda_categorical_counts_{category_col}.csv", index=False)

print("Reportes guardados en:", reports_dir)
print("Metricas guardadas en:", metrics_dir)


## 20. Conclusiones del EDA


1. El dataset presenta buena estructura general para analisis y modelado.
2. No se detectan valores nulos en el dataset original.
3. No se detectan duplicados exactos.
4. Las variables categoricas principales estan consistentes tras limpieza de texto.
5. `id_bodega` debe tratarse como categorica, no como variable continua.
6. El principal problema de limpieza esta en `target_tiempo_entrega` por registros negativos/imposibles.
7. Los tiempos menores a 10 minutos se marcan como sospechosos y no se eliminan automaticamente.
8. Los outliers del target deben revisarse con criterio de negocio antes de cualquier descarte.
9. `consumo_nafta` puede representar leakage dependiendo de su definicion operacional real.
10. Las correlaciones lineales con el target son bajas; esto no descarta relaciones no lineales o interacciones.
11. Se recomiendan features adicionales: `costo_por_km`, `consumo_por_km`, `carga_por_km`, `paradas_por_km`, `hora_sin` y `hora_cos`.
12. El dataset queda preparado para modelado supervisado y no supervisado en notebooks posteriores.


## 21. Generar docs/eda_hallazgos.md


In [ ]:
rows_count = int(df.shape[0])
cols_count = int(df.shape[1])
num_count = int(len(df.select_dtypes(include=[np.number]).columns))
cat_count = int(len(df.select_dtypes(exclude=[np.number]).columns))

invalid_count = int((pd.to_numeric(df_base["target_tiempo_entrega"], errors="coerce") <= 0).sum())
suspicious_count = int((pd.to_numeric(df_base["target_tiempo_entrega"], errors="coerce") < 10).sum())

eda_md = f'''# Hallazgos del Analisis Exploratorio de Datos

## Contexto
El proyecto analiza datos de Logistica 4.0 (Last Mile) para comprender factores asociados al tiempo de entrega y preparar el dataset para modelado posterior.

## Estructura del dataset
- Filas: {rows_count}
- Columnas: {cols_count}
- Target principal: `target_tiempo_entrega` (continua)
- Variables numericas detectadas inicialmente: {num_count}
- Variables categoricas detectadas inicialmente: {cat_count}

## Calidad de datos
- Valores nulos: 0
- Duplicados exactos: 0
- Categorias revisadas y normalizadas en trafico, clima y tipo de vehiculo.
- Reglas de negocio validadas para rangos operacionales.

## Problemas detectados
- Registros con target invalido (`<= 0`): {invalid_count}
- Registros con tiempo sospechoso (`< 10` minutos): {suspicious_count}
- Outliers detectados mediante IQR, incluyendo outliers en el target.

## Decisiones de limpieza
- Se eliminan registros con `target_tiempo_entrega <= 0` para dataset limpio.
- Los registros con `target_tiempo_entrega < 10` se conservan y se marcan con `tiempo_sospechoso`.
- `id_bodega` se convierte a categoria.
- No se eliminan outliers automaticamente sin validacion de negocio.

## Variables categoricas y encoding recomendado
- `clima`: One-Hot Encoding.
- `tipo_vehiculo`: One-Hot Encoding.
- `id_bodega`: One-Hot Encoding o tratamiento categorico.
- `trafico_nivel`: comparar encoding ordinal (Bajo=0, Medio=1, Alto=2, Critico=3) versus One-Hot.

## Posible fuga de informacion
`consumo_nafta` puede ser leakage si corresponde a consumo real posterior a la entrega. Se recomienda validar definicion y comparar modelos con/sin esta variable.

## Recomendaciones para modelado posterior
- Modelos supervisados de regresion para `target_tiempo_entrega`.
- Clasificacion derivada (por ejemplo, `entrega_tardia` o `tiempo_alto`) en experimentos separados.
- Clustering para patrones logisticos.
- Escalado con `StandardScaler` para metodos sensibles a distancia.
- Comparar experimentos con y sin `consumo_nafta`.

## Conclusion
El dataset queda preparado para la siguiente etapa de modelado, con una limpieza trazable, riesgos documentados y features exploratorias recomendadas.
'''

eda_doc_path = PROJECT_ROOT / "docs" / "eda_hallazgos.md"
eda_doc_path.write_text(eda_md, encoding="utf-8")
print("Documento generado en:", eda_doc_path)
